In [13]:
import numpy as np
import mocet

calibration_offset_start = 0.5
calibration_offset_end = -0.5
px_per_deg = 78
do_mocet = True

# Data preprocessing & Applying MoCET
subject = 'sub-006'
session = 'ses-02'
task = 'task-movieGUEST'
run = 'run-2'
root = f'../Eyetracking_data/{subject}/{session}'

if task == 'task-movieGUEST' and run == 'run-1':
    calibration_onsets = [1, 416]
    calibration_points = [24, 12]
    task_duration = 684.8
elif task == 'task-movieGUEST' and run == 'run-2':
    calibration_onsets = [1, 361]
    calibration_points = [24, 12]
    task_duration = 596.8
interval = 1.6 

log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt' 
history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
start, _, _ = mocet.get_avotec_history(history_fname)

# log, data, confound, start
pupil_data, pupil_timestamps, pupil_confidence, _ = mocet.clean_avotec_data(log_fname,
                                                                             data_fname,
                                                                             start=start,
                                                                             duration=task_duration)

pupil_validity = np.sum(np.isnan(pupil_confidence))/len(pupil_confidence)
mean_pupil_confidence = np.mean(pupil_confidence[~np.isnan(pupil_confidence)])
print(f"Eye closed: {pupil_validity:2.2f}, Avg. confidence: {mean_pupil_confidence:2.2f}")

confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'
if do_mocet:
    # Parameters
    # ----------
    # large_motion_params : bool
    #     Applying MoCET based on 24 motion parameters
    #     Default: False
    # 
    # polynomial_order : int
    #     If it is not 0, applying additional detrending using polynomial regressors
    #     Default: 0

    pupil_data = mocet.apply_mocet(pupil_data, confounds_fname,
                                   large_motion_params=False,
                                   polynomial_order=0)
    # Important note: adjust parameters if validation error is larger than 2.0 deg
    

Eye closed: 0.04, Avg. confidence: 0.99


In [14]:
calibration_coordinates = [[200, 166], [200, 500], [200, 833],
                            [600, 166], [600, 500], [600, 833],
                            [1000, 166], [1000, 500], [1000, 833],
                            [1400, 166], [1400, 500], [1400, 833]]

calibration_order = [4, 11, 6, 2, 7, 0, 10, 5, 9, 8, 1, 3]

calibration_pupils = []
calibration_t = 0 # use first 24 points for ET model calibration
offset = calibration_onsets[calibration_t]
for i in np.arange(calibration_points[calibration_t]):
    start = (offset+i)*interval + calibration_offset_start
    end = (offset+i+1)*interval + calibration_offset_end
    log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
    calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                              np.nanmean(pupil_data[log_effective,1])])
calibration_pupils = np.array(calibration_pupils)

calibrator = mocet.EyetrackingCalibration(calibration_coordinates=calibration_coordinates, 
                                          calibration_order=calibration_order,
                                          repeat=True)

calibrator.fit(calibration_pupils[:, 0], calibration_pupils[:, 1])
gaze_coordinates = calibrator.transform(pupil_data)

# Test accuracy
calibration_t = 0
validation_t = 1
for t in [calibration_t, validation_t]:
    MSE = []
    offset = calibration_onsets[t]
    for i in np.arange(calibration_points[t]):
        ref_x = calibrator.calibration_coordinates[calibrator.calibration_order[i]][0]
        ref_y = calibrator.calibration_coordinates[calibrator.calibration_order[i]][1]
        start = (offset+i)*interval + calibration_offset_start
        end = (offset+i+1)*interval + calibration_offset_end
        gaze_idx_start = np.min(np.where(pupil_timestamps >= start*1000)[0])
        gaze_idx_end = np.min(np.where(pupil_timestamps >= end*1000)[0])
        eye_closed = np.isnan(pupil_confidence[gaze_idx_start:gaze_idx_end])
        if np.any(~eye_closed):
            data_x = np.nanmean(gaze_coordinates[gaze_idx_start:gaze_idx_end,0][~eye_closed])
            data_y = np.nanmean(gaze_coordinates[gaze_idx_start:gaze_idx_end,1][~eye_closed])
            MSE.append(np.sqrt((ref_x-data_x)**2 + (ref_y-data_y)**2))
    if t == calibration_t:
        calibration_error = np.nanmean(MSE)/px_per_deg
    elif t == validation_t:
        validation_error = np.nanmean(MSE)/px_per_deg
        
print(f"[{subject}-{session}-{task}-{run}]", 
      f"Calibration error: {calibration_error:2.2f} deg, ", 
      f"Validation error: {validation_error:2.2f} deg")

np.save(f'{subject}_{session}_{task}_{run}_gaze.npy', gaze_coordinates)
np.save(f'{subject}_{session}_{task}_{run}_gaze_timestamps.npy', pupil_timestamps)

[sub-006-ses-02-task-movieGUEST-run-2] Calibration error: 0.72 deg,  Validation error: 0.87 deg


In [18]:
t = 102

gaze_idx = np.min(np.where(pupil_timestamps >= t*1000)[0])
print(gaze_coordinates[gaze_idx])